In [ ]:
%pip install tensorflow matplotlib seaborn scikit-learn numpy

In [ ]:


import os
import tarfile
import shutil
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
import tarfile


from tensorflow.keras import Sequential
from tensorflow.keras.layers import (
    Input,
    Conv2D,
    MaxPooling2D,
    Flatten,
    Dense,
    Dropout,
    BatchNormalization
)

from sklearn.metrics import classification_report, confusion_matrix

print("TensorFlow version:", tf.__version__)



In [ ]:
import os

archive_path = os.path.join(os.getcwd(), "cifar")
#https://drive.google.com/file/d/16tWgg1WB7TQMrahehq7DTCVJmaf5lJdN/view?usp=sharing
print("Archive path:", archive_path)
print("Exists:", os.path.exists(archive_path))

In [ ]:
with tarfile.open(archive_path, "r:*") as tar:
    print("Archive contents:")
    for member in tar.getnames()[:30]:
        print(member)

In [ ]:


if not os.path.isfile(archive_path):

    print("\nERROR: File not found!")
    print("\nYou entered:")
    print(archive_path)

    print("\nPlease check:")
    print("1. Did you download the CIFAR-10 file?")
    print("2. Is the path correct?")
    print("3. Did you include the complete filename?")

    raise FileNotFoundError(
        "CIFAR-10 archive was not found."
    )

else:

    file_size_mb = os.path.getsize(archive_path) / (1024 * 1024)

    print("\nCIFAR-10 archive found!")
    print("File size:", round(file_size_mb, 2), "MB")


extraction_folder = os.path.join(
    os.path.dirname(archive_path),
    "cifar10_dataset"
)

print("\nExtracting CIFAR-10 dataset...")

# Remove existing extraction folder
if os.path.exists(extraction_folder):
    shutil.rmtree(extraction_folder)

# Create a fresh folder
os.makedirs(extraction_folder)

# Extract archive
with tarfile.open(archive_path, "r:*") as tar:
    tar.extractall(path=extraction_folder, filter="data")

print("Extraction completed!")



cifar_folder = None

for root, dirs, files in os.walk(extraction_folder):

    if "data_batch_1" in files:

        cifar_folder = root
        break


if cifar_folder is None:

    raise FileNotFoundError(
        "Could not find the CIFAR-10 files after extraction."
    )


print("\nCIFAR-10 data location:")
print(cifar_folder)



print("\nLoading CIFAR-10 data...")


def load_cifar_batch(file):

    import pickle

    with open(file, "rb") as fo:

        data_dict = pickle.load(
            fo,
            encoding="bytes"
        )

    data = data_dict[b"data"]

    labels = data_dict[b"labels"]

    data = data.reshape(
        -1,
        3,
        32,
        32
    )

    data = data.transpose(
        0,
        2,
        3,
        1
    )

    return data, np.array(labels)



X_train_list = []
y_train_list = []

for i in range(1, 6):

    file_path = os.path.join(
        cifar_folder,
        f"data_batch_{i}"
    )

    X_batch, y_batch = load_cifar_batch(
        file_path
    )

    X_train_list.append(X_batch)
    y_train_list.append(y_batch)


X_train = np.concatenate(
    X_train_list,
    axis=0
)

y_train = np.concatenate(
    y_train_list,
    axis=0
)




test_file = os.path.join(
    cifar_folder,
    "test_batch"
)

X_test, y_test = load_cifar_batch(
    test_file
)


print("\nDataset loaded successfully!")

print("Training images :", X_train.shape)
print("Training labels :", y_train.shape)

print("Testing images  :", X_test.shape)
print("Testing labels  :", y_test.shape)




class_names = [
    "airplane",
    "automobile",
    "bird",
    "cat",
    "deer",
    "dog",
    "frog",
    "horse",
    "ship",
    "truck"
]




plt.figure(figsize=(10, 6))

for i in range(12):

    plt.subplot(3, 4, i + 1)

    plt.imshow(X_train[i])

    plt.title(
        class_names[y_train[i]]
    )

    plt.axis("off")

plt.suptitle(
    "Sample CIFAR-10 Images"
)

plt.tight_layout()

plt.show()




X_train = X_train.astype(
    "float32"
) / 255.0

X_test = X_test.astype(
    "float32"
) / 255.0

print("\nNormalization completed.")

print(
    "Pixel value range:",
    X_train.min(),
    "to",
    X_train.max()
)


print("\n" + "=" * 60)
print("BUILDING CNN MODEL")
print("=" * 60)


model = Sequential([

    Input(
        shape=(32, 32, 3)
    ),


    Conv2D(
        32,
        (3, 3),
        padding="same",
        activation="relu"
    ),

    BatchNormalization(),

    Conv2D(
        32,
        (3, 3),
        padding="same",
        activation="relu"
    ),

    MaxPooling2D(
        pool_size=(2, 2)
    ),

    Dropout(0.25),


 

    Conv2D(
        64,
        (3, 3),
        padding="same",
        activation="relu"
    ),

    BatchNormalization(),

    Conv2D(
        64,
        (3, 3),
        padding="same",
        activation="relu"
    ),

    MaxPooling2D(
        pool_size=(2, 2)
    ),

    Dropout(0.25),


  

    Flatten(),

    Dense(
        128,
        activation="relu"
    ),

    Dropout(0.5),

    Dense(
        10,
        activation="softmax"
    )
])



model.summary()



model.compile(

    optimizer="adam",

    loss="sparse_categorical_crossentropy",

    metrics=["accuracy"]
)

print("\nModel compiled successfully.")


print("\n" + "=" * 60)
print("TRAINING CNN")
print("=" * 60)

history = model.fit(

    X_train,

    y_train,

    epochs=5,

    batch_size=128,

    validation_split=0.1,

    verbose=1
)



print("\n" + "=" * 60)
print("MODEL EVALUATION")
print("=" * 60)

test_loss, test_accuracy = model.evaluate(

    X_test,

    y_test,

    verbose=0
)


print(
    "Test Loss:",
    round(test_loss, 4)
)

print(
    "Test Accuracy:",
    round(test_accuracy, 4)
)

print(
    "Test Accuracy (%):",
    round(test_accuracy * 100, 2),
    "%"
)




plt.figure(figsize=(8, 5))

plt.plot(
    history.history["accuracy"],
    label="Training Accuracy"
)

plt.plot(
    history.history["val_accuracy"],
    label="Validation Accuracy"
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy")

plt.title(
    "Training and Validation Accuracy"
)

plt.legend()

plt.grid(True)

plt.show()



plt.figure(figsize=(8, 5))

plt.plot(
    history.history["loss"],
    label="Training Loss"
)

plt.plot(
    history.history["val_loss"],
    label="Validation Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")

plt.title(
    "Training and Validation Loss"
)

plt.legend()

plt.grid(True)

plt.show()




print("\nGenerating predictions...")

y_pred = model.predict(

    X_test,

    verbose=0
)

y_pred_classes = np.argmax(

    y_pred,

    axis=1
)

print(
    "Predictions generated successfully."
)




print("\n" + "=" * 60)
print("CLASSIFICATION REPORT")
print("=" * 60)

print(

    classification_report(

        y_test,

        y_pred_classes,

        target_names=class_names

    )

)



cm = confusion_matrix(

    y_test,

    y_pred_classes

)


plt.figure(
    figsize=(10, 8)
)

sns.heatmap(

    cm,

    annot=True,

    fmt="d",

    xticklabels=class_names,

    yticklabels=class_names

)

plt.xlabel(
    "Predicted Label"
)

plt.ylabel(
    "Actual Label"
)

plt.title(
    "CIFAR-10 Confusion Matrix"
)

plt.show()




plt.figure(
    figsize=(12, 8)
)

for i in range(12):

    plt.subplot(
        3,
        4,
        i + 1
    )

    plt.imshow(
        X_test[i]
    )

    actual = class_names[
        y_test[i]
    ]

    predicted = class_names[
        y_pred_classes[i]
    ]

    plt.title(
        f"Actual: {actual}\n"
        f"Predicted: {predicted}",
        fontsize=9
    )

    plt.axis("off")


plt.suptitle(
    "CIFAR-10 Predictions"
)

plt.tight_layout()

plt.show()



In [ ]:

print("\nExtracting feature maps...")

# Use the model's explicit Input layer
feature_model = tf.keras.Model(
    inputs=model.layers[0].input,
    outputs=model.layers[0].output
)

sample_image = X_test[0:1]

feature_maps = feature_model.predict(
    sample_image,
    verbose=0
)

print("Feature map shape:", feature_maps.shape)


plt.figure(
    figsize=(12, 8)
)

for i in range(8):

    plt.subplot(
        2,
        4,
        i + 1
    )

    plt.imshow(

        feature_maps[0, :, :, i],

        cmap="viridis"

    )

    plt.title(
        f"Filter {i + 1}"
    )

    plt.axis("off")


plt.suptitle(
    "Feature Maps from First Convolution Layer"
)

plt.tight_layout()

plt.show()



index = 100

image = X_test[index]


prediction = model.predict(

    image.reshape(
        1,
        32,
        32,
        3
    ),

    verbose=0

)


predicted_class = np.argmax(
    prediction
)


confidence = (
    float(
        np.max(prediction)
    ) * 100
)


print("\n" + "=" * 60)
print("SINGLE IMAGE PREDICTION")
print("=" * 60)

print(
    "Actual Class:",
    class_names[y_test[index]]
)

print(
    "Predicted Class:",
    class_names[predicted_class]
)

print(
    "Confidence:",
    round(
        confidence,
        2
    ),
    "%"
)


plt.figure(
    figsize=(5, 5)
)

plt.imshow(
    image
)

plt.title(

    f"Actual: {class_names[y_test[index]]}\n"
    f"Predicted: {class_names[predicted_class]}\n"
    f"Confidence: {confidence:.2f}%"

)

plt.axis("off")

plt.show()



print("\n" + "=" * 60)
print("EXPERIMENT 3A COMPLETED")
print("=" * 60)

print(
    "Dataset       : CIFAR-10"
)

print(
    "Training Data :",
    X_train.shape
)

print(
    "Testing Data  :",
    X_test.shape
)

print(
    "Test Accuracy :",
    round(
        test_accuracy * 100,
        2
    ),
    "%"
)

print(
    "Evaluation    : Classification Report + Confusion Matrix"
)

print(
    "Visualization : Feature Maps"
)

print("\nExperiment completed successfully!")